In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['RSI_Strategy2']
trades_collection  = db['trades_connors_rsi2']
exclude_collection = db['excluded_tickers_connors_rsi2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

MA_PERIOD        = 200    # Trend filter: 200-day SMA
RSI2_PERIOD      = 2      # Connors RSI period
RSI2_OVERSOLD    = 5      # Long entry: RSI(2) < 5
RSI2_OVERBOUGHT  = 95     # Short entry: RSI(2) > 95
RSI2_LONG_EXIT   = 70     # Long exit: RSI(2) > 70
RSI2_SHORT_EXIT  = 50     # Short exit: RSI(2) < 50
ORDER_SIZE       = 40     # Fixed shares per trade


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER  — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    This prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order     = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series, period=2):
    """Compute RSI for a given period on a price Series."""
    d    = series.diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    return 100 - 100 / (1 + gain / (loss + 1e-10))


def calc_indicators(df, ma_period=MA_PERIOD, rsi_period=RSI2_PERIOD):
    """
    Add 200-day SMA (trend filter) and RSI(2) (entry/exit signal) to df.

    NOTE: The 200-day MA requires daily bars.  We request daily bars
    separately inside check_and_trade() and attach the result here.
    """
    df['RSI2'] = calc_rsi(df['close'], period=rsi_period)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_ma(contract, period=MA_PERIOD):
    """
    Fetch enough daily bars to compute the 200-day SMA.
    Returns the latest SMA value and the latest closing price.
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{period + 10} D',   # a little extra buffer
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < period:
        return None, None

    df        = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    sma       = float(df['close'].rolling(period).mean().iloc[-1])
    last_close = float(df['close'].iloc[-1])
    return sma, last_close


def get_intraday_rsi2(contract):
    """
    Fetch 5-min intraday bars and compute RSI(2).
    Returns (price_now, rsi2_now, rsi2_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='3 D',           # 3 days of 5-min bars for warm-up
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < 10:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df         = calc_indicators(df)

    price     = float(df.iloc[-1]['close'])
    rsi2_now  = float(df.iloc[-1]['RSI2'])
    rsi2_prev = float(df.iloc[-2]['RSI2'])
    return price, rsi2_now, rsi2_prev


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi2_at_entry, sma200):
    return {
        'instrument':      symbol,
        'direction':       direction,            # 'long' or 'short'
        'entry_price':     float(entry_price),
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi2_at_entry':   float(rsi2_at_entry),
        'sma200_at_entry': float(sma200),
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi2':       None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }


def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})


def do_sell(contract, qty, entry_price, sell_price, rsi2, reason, trade_id):
    """Execute market sell (close long) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl  = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


def do_cover(contract, qty, entry_price, cover_price, rsi2, reason, trade_id):
    """Execute market buy (cover short) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl  = (entry_price - cover_price) * qty   # short PnL is reversed
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  CONNORS RSI(2) MEAN-REVERSION STRATEGY                                 ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  TREND FILTER  — 200-day SMA (daily bars)                               ║
    ║    Price > SMA200 → bullish bias  → longs only                          ║
    ║    Price < SMA200 → bearish bias  → shorts only                         ║
    ║                                                                          ║
    ║  ENTRY (5-min RSI2)                                                      ║
    ║    Long  — RSI(2) < 5   (strongly oversold dip inside uptrend)          ║
    ║    Short — RSI(2) > 95  (strongly overbought spike inside downtrend)    ║
    ║                                                                          ║
    ║  EXIT                                                                    ║
    ║    Long  — RSI(2) > 70  (mean-reversion complete)                       ║
    ║    Short — RSI(2) < 50  (mean-reversion complete)                       ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── 1. Trend filter: 200-day SMA ──────────────────────────────
            sma200, daily_close = get_daily_ma(contract)
            if sma200 is None:
                print(f"  Insufficient daily data for SMA200 — skipping {symbol}")
                continue

            price_above_sma = daily_close > sma200
            trend_label     = 'UPTREND ▲' if price_above_sma else 'DOWNTREND ▼'
            print(f"  SMA200=${sma200:.2f}  DailyClose=${daily_close:.2f}  {trend_label}")

            # ── 2. RSI(2) on 5-min bars ────────────────────────────────────
            price, rsi2_now, rsi2_prev = get_intraday_rsi2(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                continue

            print(f"  Price=${price:.2f}  RSI2={rsi2_now:.1f}  RSI2_prev={rsi2_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                direction   = open_trade.get('direction', 'long')
                trade_id    = open_trade['_id']

                if direction == 'long':
                    current_pnl = (price - entry_price) * qty
                    profit_pct  = (price - entry_price) / entry_price
                    print(f"  OPEN LONG  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit long when RSI(2) > 70
                    if rsi2_now > RSI2_LONG_EXIT:
                        do_sell(contract, qty, entry_price, price,
                                rsi2_now, 'RSI2_ABOVE_70_LONG_EXIT', trade_id)
                        print(f"  📈 LONG EXIT: RSI2={rsi2_now:.1f} > {RSI2_LONG_EXIT}")
                    else:
                        print(f"  Holding long — waiting RSI2 > {RSI2_LONG_EXIT}  (RSI2={rsi2_now:.1f})")

                elif direction == 'short':
                    current_pnl = (entry_price - price) * qty
                    profit_pct  = (entry_price - price) / entry_price
                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit short when RSI(2) < 50
                    if rsi2_now < RSI2_SHORT_EXIT:
                        do_cover(contract, qty, entry_price, price,
                                 rsi2_now, 'RSI2_BELOW_50_SHORT_EXIT', trade_id)
                        print(f"  📉 SHORT EXIT: RSI2={rsi2_now:.1f} < {RSI2_SHORT_EXIT}")
                    else:
                        print(f"  Holding short — waiting RSI2 < {RSI2_SHORT_EXIT}  (RSI2={rsi2_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                # ── LONG: uptrend + RSI(2) oversold ──────────────────────
                if price_above_sma and rsi2_now < RSI2_OVERSOLD:
                    ib.placeOrder(contract, day_market_order('BUY', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'long', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY (long)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (above)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (oversold dip < {RSI2_OVERSOLD})")

                # ── SHORT: downtrend + RSI(2) overbought ─────────────────
                elif not price_above_sma and rsi2_now > RSI2_OVERBOUGHT:
                    ib.placeOrder(contract, day_market_order('SELL', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'short', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🔻 SELL (short)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (below)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (overbought spike > {RSI2_OVERBOUGHT})")

                else:
                    bias = 'long' if price_above_sma else 'short'
                    threshold = f'< {RSI2_OVERSOLD}' if price_above_sma else f'> {RSI2_OVERBOUGHT}'
                    print(f"  No entry — bias={bias}  RSI2={rsi2_now:.1f}"
                          f"  (waiting for RSI2 {threshold})")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  07:20:49
  SMA200=$108.54  DailyClose=$122.18  UPTREND ▲
  Price=$122.00  RSI2=58.5  RSI2_prev=58.5
  OPEN LONG  entry=$122.42  qty=40  P&L=-0.34%  ($-16.80)
  Holding long — waiting RSI2 > 70  (RSI2=58.5)

  MU  07:20:50
  SMA200=$237.25  DailyClose=$355.46  UPTREND ▲
  Price=$347.80  RSI2=67.9  RSI2_prev=57.4
  No entry — bias=long  RSI2=67.9  (waiting for RSI2 < 5)


Error 200, reqId 78: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  07:20:50
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  07:20:50
  SMA200=$5.79  DailyClose=$7.73  UPTREND ▲
  Price=$7.73  RSI2=0.0  RSI2_prev=0.0
  OPEN LONG  entry=$7.73  qty=40  P&L=+0.00%  ($+0.00)
  Holding long — waiting RSI2 > 70  (RSI2=0.0)

  SLV  07:20:51
  SMA200=$52.02  DailyClose=$60.77  UPTREND ▲
  Price=$61.62  RSI2=17.3  RSI2_prev=10.9
  No entry — bias=long  RSI2=17.3  (waiting for RSI2 < 5)

  NEM  07:20:52
  SMA200=$88.83  DailyClose=$99.36  UPTREND ▲
  Price=$99.34  RSI2=81.6  RSI2_prev=81.6
  No entry — bias=long  RSI2=81.6  (waiting for RSI2 < 5)

  CTXR  07:20:53
  SMA200=$1.16  DailyClose=$0.69  DOWNTREND ▼
  Price=$0.70  RSI2=6.0  RSI2_prev=6.0
  No entry — bias=short  RSI2=6.0  (waiting for RSI2 > 95)

  ONON  07:20:54
  SMA200=$45.56  DailyClose=$32.11  DOWNTREND ▼
  Price=$31.95  RSI2=52.1  RSI2_prev=52.1
  No entry — bias=short  RSI2=52.1  (waiting for RSI2 > 95)

  IONQ  07:20:55
  SMA200=$47.05  DailyClose=$29.84  DOWNTREND ▼
  Pr

Error 200, reqId 131: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


...


  BRK.B  10:54:58
  Insufficient daily data for SMA200 — skipping BRK.B

  IBM  10:54:58
  SMA200=$278.32  DailyClose=$236.00  DOWNTREND ▼
  Price=$236.00  RSI2=37.0  RSI2_prev=86.9
  Already traded IBM today — skipping.

  MSFT  10:54:58
  SMA200=$479.10  DailyClose=$359.33  DOWNTREND ▼
  Price=$359.33  RSI2=28.5  RSI2_prev=99.2
  OPEN SHORT  entry=$359.24  qty=40  P&L=-0.03%  ($-3.60)
  ✅ COVER (close short) 40sh @ $359.33  RSI2=28.5  [RSI2_BELOW_50_SHORT_EXIT]  PnL: $-3.60
  📉 SHORT EXIT: RSI2=28.5 < 50

  KO  10:54:59
  SMA200=$71.20  DailyClose=$75.39  UPTREND ▲
  Price=$75.39  RSI2=10.1  RSI2_prev=23.6
  No entry — bias=long  RSI2=10.1  (waiting for RSI2 < 5)

  MSTR  10:55:00
  SMA200=$259.53  DailyClose=$125.48  DOWNTREND ▼
  Price=$125.48  RSI2=87.9  RSI2_prev=83.0
  Already traded MSTR today — skipping.

  COIN  10:55:00
  SMA200=$281.92  DailyClose=$161.37  DOWNTREND ▼
  Price=$161.37  RSI2=45.4  RSI2_prev=82.3
  No entry — bias=short  RSI2=45.4  (waiting for RSI2 > 95)

  

Error 200, reqId 5500: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  11:00:22
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  11:00:22
  SMA200=$5.80  DailyClose=$7.36  UPTREND ▲
  Price=$7.36  RSI2=65.8  RSI2_prev=65.8
  No entry — bias=long  RSI2=65.8  (waiting for RSI2 < 5)

  SLV  11:00:23
  SMA200=$52.17  DailyClose=$62.48  UPTREND ▲
  Price=$62.48  RSI2=81.5  RSI2_prev=65.9
  Already traded SLV today — skipping.

  NEM  11:00:23
  SMA200=$89.08  DailyClose=$102.05  UPTREND ▲
  Price=$102.05  RSI2=99.2  RSI2_prev=99.1
  No entry — bias=long  RSI2=99.2  (waiting for RSI2 < 5)

  CTXR  11:00:24
  SMA200=$1.16  DailyClose=$0.70  DOWNTREND ▼
  Price=$0.70  RSI2=69.2  RSI2_prev=69.2
  Already traded CTXR today — skipping.

  ONON  11:00:25
  SMA200=$45.44  DailyClose=$32.63  DOWNTREND ▼
  Price=$32.62  RSI2=23.8  RSI2_prev=26.6
  OPEN SHORT  entry=$32.23  qty=40  P&L=-1.21%  ($-15.60)
  ✅ COVER (close short) 40sh @ $32.62  RSI2=23.8  [RSI2_BELOW_50_SHORT_EXIT]  PnL: $-15.60
  📉 SHORT EXIT: RSI2=23.8 < 50

  IONQ  11:00:25
  SMA200

Error 200, reqId 5552: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  BRK.B  11:00:35
  Insufficient daily data for SMA200 — skipping BRK.B

  IBM  11:00:35
  SMA200=$278.32  DailyClose=$236.47  DOWNTREND ▼
  Price=$236.47  RSI2=75.8  RSI2_prev=69.9
  Already traded IBM today — skipping.

  MSFT  11:00:36
  SMA200=$479.10  DailyClose=$360.05  DOWNTREND ▼
  Price=$360.05  RSI2=68.2  RSI2_prev=57.3
  Already traded MSFT today — skipping.

  KO  11:00:36
  SMA200=$71.20  DailyClose=$75.53  UPTREND ▲
  Price=$75.53  RSI2=82.0  RSI2_prev=82.0
  No entry — bias=long  RSI2=82.0  (waiting for RSI2 < 5)

  MSTR  11:00:37
  SMA200=$259.53  DailyClose=$125.73  DOWNTREND ▼
  Price=$125.73  RSI2=94.2  RSI2_prev=92.5
  Already traded MSTR today — skipping.

  COIN  11:00:37
  SMA200=$281.92  DailyClose=$161.55  DOWNTREND ▼
  Price=$161.55  RSI2=67.1  RSI2_prev=61.5
  No entry — bias=short  RSI2=67.1  (waiting for RSI2 > 95)

  GLD  11:00:38
  SMA200=$376.62  DailyClose=$410.43  UPTREND ▲
  Price=$410.43  RSI2=83.4  RSI2_prev=68.9
  No entry — bias=long  RSI2=83.4  